In [1]:
import pandas as pd

# Load datasets
depression = pd.read_csv("/content/depression_dataset_reddit_cleaned.csv")
stress_train = pd.read_csv("/content/dreaddit-train.csv")
stress_test = pd.read_csv("/content/dreaddit-test.csv")

# Check dataset shapes
print("Depression shape:", depression.shape)
print("Stress train shape:", stress_train.shape)
print("Stress test shape:", stress_test.shape)

# Check column names
print("\nDepression columns:")
print(depression.columns.tolist())

print("\nStress train columns:")
print(stress_train.columns.tolist())

print("\nStress test columns:")
print(stress_test.columns.tolist())

Depression shape: (7731, 2)
Stress train shape: (2838, 116)
Stress test shape: (715, 116)

Depression columns:
['clean_text', 'is_depression']

Stress train columns:
['subreddit', 'post_id', 'sentence_range', 'text', 'id', 'label', 'confidence', 'social_timestamp', 'social_karma', 'syntax_ari', 'lex_liwc_WC', 'lex_liwc_Analytic', 'lex_liwc_Clout', 'lex_liwc_Authentic', 'lex_liwc_Tone', 'lex_liwc_WPS', 'lex_liwc_Sixltr', 'lex_liwc_Dic', 'lex_liwc_function', 'lex_liwc_pronoun', 'lex_liwc_ppron', 'lex_liwc_i', 'lex_liwc_we', 'lex_liwc_you', 'lex_liwc_shehe', 'lex_liwc_they', 'lex_liwc_ipron', 'lex_liwc_article', 'lex_liwc_prep', 'lex_liwc_auxverb', 'lex_liwc_adverb', 'lex_liwc_conj', 'lex_liwc_negate', 'lex_liwc_verb', 'lex_liwc_adj', 'lex_liwc_compare', 'lex_liwc_interrog', 'lex_liwc_number', 'lex_liwc_quant', 'lex_liwc_affect', 'lex_liwc_posemo', 'lex_liwc_negemo', 'lex_liwc_anx', 'lex_liwc_anger', 'lex_liwc_sad', 'lex_liwc_social', 'lex_liwc_family', 'lex_liwc_friend', 'lex_liwc_female

In [2]:
import pandas as pd

# -----------------------------------
# 1. Combine Dreaddit train + test
# -----------------------------------
stress_all = pd.concat(
    [stress_train, stress_test],
    ignore_index=True
)

print("Total Dreaddit samples:", len(stress_all))

# -----------------------------------
# 2. Select only Stress = 1
# -----------------------------------
stress = stress_all[stress_all["label"] == 1][["text"]].copy()
stress["label"] = "Stress"

# -----------------------------------
# 3. Select only Depression = 1
# -----------------------------------
depression_positive = depression[
    depression["is_depression"] == 1
][["clean_text"]].copy()

depression_positive = depression_positive.rename(
    columns={"clean_text": "text"}
)

depression_positive["label"] = "Depression"

# -----------------------------------
# 4. Remove missing / empty text
# -----------------------------------
stress = stress.dropna(subset=["text"])
depression_positive = depression_positive.dropna(subset=["text"])

stress["text"] = stress["text"].astype(str).str.strip()
depression_positive["text"] = (
    depression_positive["text"].astype(str).str.strip()
)

stress = stress[stress["text"] != ""]
depression_positive = depression_positive[
    depression_positive["text"] != ""
]

# -----------------------------------
# 5. Remove duplicates within classes
# -----------------------------------
stress_before = len(stress)
depression_before = len(depression_positive)

stress = stress.drop_duplicates(subset=["text"])
depression_positive = depression_positive.drop_duplicates(
    subset=["text"]
)

print("\n--- BEFORE BALANCING ---")
print("Stress:", len(stress))
print("Depression:", len(depression_positive))

print("\nDuplicates removed:")
print("Stress:", stress_before - len(stress))
print(
    "Depression:",
    depression_before - len(depression_positive)
)

# -----------------------------------
# 6. Check overlap between classes
# -----------------------------------
overlap = set(stress["text"]).intersection(
    set(depression_positive["text"])
)

print("\nExact text appearing in BOTH classes:", len(overlap))

# Remove cross-class duplicates if any
if len(overlap) > 0:
    stress = stress[~stress["text"].isin(overlap)]
    depression_positive = depression_positive[
        ~depression_positive["text"].isin(overlap)
    ]

# -----------------------------------
# 7. Balance classes
# -----------------------------------
n = min(len(stress), len(depression_positive))

stress_balanced = stress.sample(
    n=n,
    random_state=42
)

depression_balanced = depression_positive.sample(
    n=n,
    random_state=42
)

# -----------------------------------
# 8. Combine and shuffle
# -----------------------------------
final_data = pd.concat(
    [stress_balanced, depression_balanced],
    ignore_index=True
)

final_data = final_data.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# -----------------------------------
# 9. Display final results
# -----------------------------------
print("\n--- FINAL DATASET ---")
print(final_data["label"].value_counts())

print("\nTotal final samples:", len(final_data))

print("\nFirst 5 rows:")
display(final_data.head())

# -----------------------------------
# 10. Save final curated dataset
# -----------------------------------
final_data.to_csv(
    "/content/final_stress_depression_dataset.csv",
    index=False
)

print("\nSaved: final_stress_depression_dataset.csv")

Total Dreaddit samples: 3553

--- BEFORE BALANCING ---
Stress: 1854
Depression: 3761

Duplicates removed:
Stress: 3
Depression: 70

Exact text appearing in BOTH classes: 0

--- FINAL DATASET ---
label
Stress        1854
Depression    1854
Name: count, dtype: int64

Total final samples: 3708

First 5 rows:


,text,label
0,I have anxiety ptsd depression and a severe ea...,Stress
1,idk if anyone else get this feeling but i do a...,Depression
2,i m really desperate i m a yr old guy with no ...,Depression
3,And I hated him too. I had dreams of him stabb...,Stress
4,My mom is a single mom working two shifts ever...,Stress



Saved: final_stress_depression_dataset.csv


In [3]:
import matplotlib.pyplot as plt

# Word count for each post
final_data["word_count"] = final_data["text"].astype(str).apply(
    lambda x: len(x.split())
)

# Separate classes
stress_lengths = final_data[
    final_data["label"] == "Stress"
]["word_count"]

depression_lengths = final_data[
    final_data["label"] == "Depression"
]["word_count"]

# Print statistics
print("=== STRESS TEXT LENGTH ===")
print(stress_lengths.describe())

print("\n=== DEPRESSION TEXT LENGTH ===")
print(depression_lengths.describe())

print("\nMean word count:")
print("Stress:", round(stress_lengths.mean(), 2))
print("Depression:", round(depression_lengths.mean(), 2))

print("\nMedian word count:")
print("Stress:", stress_lengths.median())
print("Depression:", depression_lengths.median())

=== STRESS TEXT LENGTH ===
count    1854.000000
mean       88.861920
std        34.194838
min         1.000000
25%        67.000000
50%        83.000000
75%       105.000000
max       310.000000
Name: word_count, dtype: float64

=== DEPRESSION TEXT LENGTH ===
count    1854.000000
mean      138.480583
std       172.786391
min         1.000000
25%        35.000000
50%        84.000000
75%       179.000000
max      1747.000000
Name: word_count, dtype: float64

Mean word count:
Stress: 88.86
Depression: 138.48

Median word count:
Stress: 83.0
Depression: 84.0
